In [123]:
print("Test")

Test


In [114]:
import pathlib
import random
import numpy as np

from scripts.config import (
    SEED,
    ROOT,
    DATA_DIR,
    ENC2017_ROOT,
    UD_ET_EDT_ROOT,
    HOMONYMS_ROOT,
    ENC2017_DIRS,
    UD_ET_EDT_DIRS,
    HOMONYMS_DIRS,
    OUTPUT_DIR,
    MODEL_DIR,
)

random_seed = SEED
random.seed(SEED)
np.random.seed(SEED)

In [3]:
import pandas as pd
import ast
import tqdm
import glob
import sys
import estnltk
from estnltk.default_resolver import make_resolver
from estnltk.taggers import VabamorfAnalyzer
from estnltk_neural.taggers import StanzaSyntaxTagger

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [124]:
import importlib
import scripts.DepChainTagger

# Reload the module to ensure that the latest changes are reflected
importlib.reload(scripts.DepChainTagger);

In [4]:
# Download the Stanza models for Estonian if not already present
stanza_syntax_tagger = StanzaSyntaxTagger(
    input_type="morph_analysis",
    input_morph_layer="morph_analysis",
    add_parent_and_children=True,
)
# estnltk.download('stanzasyntaxtagger')

In [5]:
# sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
sample_text = "Üks ütles, et 1. mail tähistab palju maid töörahvapüha."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

Text(text='Üks ütles, et 1. mail tähistab palju maid töörahvapüha.')

### Unit tests


In [6]:
# SyntaxGraphIndex class testing
from types import SimpleNamespace
from typing import Type

from scripts.DepChainTagger import SyntaxGraphIndex


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling `fn(*args, **kwargs)` raises `exc_type`."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def make_ann(
    token_id: int,
    head: int,
    text: str,
    upostag: str = "NOUN",
    xpostag: str = "S",
    deprel: str = "nmod",
    lemma: str = "_",
    feats: dict | None = None,
) -> SimpleNamespace:
    """Create a lightweight stanza-like annotation object for unit tests."""
    return SimpleNamespace(
        id=token_id,
        head=head,
        text=text,
        upostag=upostag,
        xpostag=xpostag,
        deprel=deprel,
        lemma=lemma,
        feats={} if feats is None else feats,
    )


def run_syntaxgraphindex_tests() -> None:
    """Run behavioural and integrity tests for SyntaxGraphIndex."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one test condition and raise with readable context on failure."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # 1) Happy path: simple three-token tree
    layer_ok = [
        make_ann(1, 0, "root", deprel="root"),
        make_ann(2, 1, "left_child", deprel="nmod"),
        make_ann(3, 1, "right_child", deprel="obl"),
    ]
    graph = SyntaxGraphIndex(layer_ok, sentence_id=0, sentence_span=(0, 15))

    check(graph.sent_id == 0, "sent_id should be preserved.")
    check(graph.sentence_span == (0, 15), "sentence_span should be preserved.")
    check(graph.token_order == [1, 2, 3], "token_order should match input order.")
    check(graph.has_node(2), "has_node should return True for existing node.")
    check(not graph.has_node(99), "has_node should return False for missing node.")
    check(graph.get_node(1).text == "root", "get_node should return correct node.")
    check(graph.get_parent(1) is None, "Root node parent should be None.")
    check(graph.get_parent(2).id == 1, "Child parent should be the root node.")

    child_ids_of_1 = [node.id for node in graph.get_children(1)]
    check(
        child_ids_of_1 == [2, 3],
        "get_children should return children in insertion order.",
    )
    check(
        [node.id for node in graph.get_root_nodes()] == [1],
        "Exactly one root node (id=1) is expected.",
    )
    check(graph._validate_tree(), "Valid tree should pass validation.")

    # iter_nodes should expose nodes in token order
    check(
        [node.id for node in graph.iter_nodes()] == [1, 2, 3],
        "iter_nodes should follow token_order.",
    )

    # iter_edges should expose expected directional pairs
    edges_up = [
        (node.id, parent.id, direction)
        for node, parent, direction in graph.iter_edges("up")
    ]
    check(
        edges_up == [(2, 1, "up"), (3, 1, "up")],
        "iter_edges('up') should contain child->parent edges only.",
    )

    edges_down = [
        (node.id, child.id, direction)
        for node, child, direction in graph.iter_edges("down")
    ]
    check(
        edges_down == [(1, 2, "down"), (1, 3, "down")],
        "iter_edges('down') should contain parent->child edges only.",
    )

    # 2) Integrity: duplicate token IDs should fail fast
    layer_duplicate_ids = [
        make_ann(1, 0, "root"),
        make_ann(1, 1, "duplicate"),
    ]
    assert_raises(ValueError, SyntaxGraphIndex, layer_duplicate_ids)
    passed += 1
    total += 1

    # 3) Integrity: missing head reference should fail fast
    layer_missing_head = [
        make_ann(1, 0, "root"),
        make_ann(2, 99, "orphan"),
    ]
    assert_raises(ValueError, SyntaxGraphIndex, layer_missing_head)
    passed += 1
    total += 1

    # 4) Integrity: rootless/cyclic structure should fail validation
    layer_cycle = [
        make_ann(1, 2, "a"),
        make_ann(2, 1, "b"),
    ]
    assert_raises(ValueError, SyntaxGraphIndex, layer_cycle)
    passed += 1
    total += 1

    print(f"SyntaxGraphIndex tests passed: {passed}/{total}")


run_syntaxgraphindex_tests()

SyntaxGraphIndex tests passed: 17/17


In [7]:
# Testing SyntaxGraphIndex on sample text with Stanza syntax annotations

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the SyntaxGraphIndex implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}, Token Order: {graph_index.token_order}"
    )
    print("Nodes in the graph:")
    for node in graph_index.iter_nodes():
        print(
            f"Node ID: {node.id}, Text: '{node.text}', UPOS: {node.upostag}, XPOS: {node.xpostag}, Deprel: {node.deprel}, Head: {node.head}"
        )
    print("\nEdges in the graph:")
    for node, parent, direction in graph_index.iter_edges():
        parent_text = parent.text if parent else "None"
        parent_id = parent.id if parent else "None"
        if node and parent:
            print(
                f"Node ID: {node.id} ('{node.text}') --[{direction}]--> Parent ID: {parent_id} ('{parent_text}')"
            )
    print("\nRoot nodes in the graph:")
    root_nodes = graph_index.get_root_nodes()
    for root in root_nodes:
        print(f"Root Node ID: {root.id}, Text: '{root.text}'")
    print("\nValidating tree structure...")
    is_valid_tree = graph_index._validate_tree()
    print(f"Is the graph a valid tree? {is_valid_tree}")
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42), Token Order: [1, 2, 3, 4, 5, 6, 7]
Nodes in the graph:
Node ID: 1, Text: 'Ta', UPOS: P, XPOS: P, Deprel: nsubj, Head: 2
Node ID: 2, Text: 'andis', UPOS: V, XPOS: V, Deprel: root, Head: 0
Node ID: 3, Text: 'lendurist', UPOS: S, XPOS: S, Deprel: nmod, Head: 4
Node ID: 4, Text: 'abikaasale', UPOS: S, XPOS: S, Deprel: obj, Head: 2
Node ID: 5, Text: 'oma', UPOS: P, XPOS: P, Deprel: nmod, Head: 6
Node ID: 6, Text: 'raamatu', UPOS: S, XPOS: S, Deprel: obj, Head: 2
Node ID: 7, Text: '.', UPOS: Z, XPOS: Z, Deprel: punct, Head: 2

Edges in the graph:
Node ID: 1 ('Ta') --[up]--> Parent ID: 2 ('andis')
Node ID: 2 ('andis') --[down]--> Parent ID: 1 ('Ta')
Node ID: 3 ('lendurist') --[up]--> Parent ID: 4 ('abikaasale')
Node ID: 4 ('abikaasale') --[down]--> Parent ID: 3 ('lendurist')
Node ID: 4 ('abikaasale') --[up]--> Parent ID: 2 ('andis')
Node ID: 2 ('andis') --[down]--> Parent ID: 4 ('abikaasale')
Node ID: 5 ('oma') --[up]--> Parent ID: 6 ('raamatu')
Node ID

In [8]:
# ValueCondition class testing
from typing import Any, Type
from scripts.DepChainTagger import ValueCondition, ConditionMode


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling `fn(*args, **kwargs)` raises `exc_type`."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def run_valuecondition_tests() -> None:
    """Run focused behavioural tests for ValueCondition and print a compact summary."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one test condition and raise with a readable message on failure."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # 1) EXACT mode basic behaviour
    exact = ValueCondition(mode=ConditionMode.EXACT, value="NOUN")
    check(exact.matches("NOUN"), "EXACT should match equal value.")
    check(not exact.matches("VERB"), "EXACT should reject different value.")

    # 2) NEGATION mode basic behaviour
    neg = ValueCondition(mode=ConditionMode.NEGATION, value="VERB")
    check(neg.matches("NOUN"), "NEGATION should match non-equal value.")
    check(not neg.matches("VERB"), "NEGATION should reject equal value.")

    # 3) WILDCARD mode should match everything
    wildcard = ValueCondition(mode=ConditionMode.WILDCARD, value=None)
    check(wildcard.matches("anything"), "WILDCARD should match non-missing values.")
    check(wildcard.matches(None), "WILDCARD should match missing values too.")

    # 4) Missing-value handling in EXACT/NEGATION
    exact_missing = ValueCondition(
        mode=ConditionMode.EXACT,
        value="NOUN",
        allow_missing=True,
    )
    check(exact_missing.matches(None), "allow_missing=True should accept None.")
    check(
        not exact.matches(None),
        "allow_missing=False should reject missing values in EXACT.",
    )

    # 5) Normaliser should pre-normalise expected and normalise actual
    exact_norm = ValueCondition(
        mode=ConditionMode.EXACT,
        value="Noun",
        normalizer=lambda x: x.lower() if isinstance(x, str) else x,
    )
    check(
        exact_norm.matches("NOUN"),
        "Normaliser should make comparison case-insensitive here.",
    )

    # 6) Constructor validation checks
    assert_raises(
        ValueError,
        ValueCondition,
        mode=ConditionMode.EXACT,
        value=None,
    )
    assert_raises(
        ValueError,
        ValueCondition,
        mode=ConditionMode.WILDCARD,
        value="not-none",
    )
    assert_raises(
        TypeError,
        ValueCondition,
        mode=ConditionMode.EXACT,
        value="NOUN",
        normalizer="not-callable",
    )
    passed += 3
    total += 3

    print(f"ValueCondition tests passed: {passed}/{total}")


run_valuecondition_tests()

ValueCondition tests passed: 12/12


In [9]:
# Testing ValueCondition on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the ValueCondition implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing ValueCondition on nodes:")
    for node in graph_index.iter_nodes():
        node_upostag = node.upostag if node.upostag else "None"
        condition_exact = ValueCondition(
            mode=ConditionMode.EXACT,
            value="S",
        )
        condition_negation = ValueCondition(
            mode=ConditionMode.NEGATION,
            value="S",
        )
        condition_wildcard = ValueCondition(
            mode=ConditionMode.WILDCARD,
        )
        result_exact = condition_exact.matches(node_upostag)
        result_negation = condition_negation.matches(node_upostag)
        result_wildcard = condition_wildcard.matches(node_upostag)
        print(f"Node ID: {node.id}, Text: '{node.text}', UPOS: {node.upostag}")
        print(f"  ValueCondition(EXACT, 'S') matches: {result_exact}")
        print(f"  ValueCondition(NEGATION, 'S') matches: {result_negation}")
        print(f"  ValueCondition(WILDCARD) matches: {result_wildcard}")
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing ValueCondition on nodes:
Node ID: 1, Text: 'Ta', UPOS: P
  ValueCondition(EXACT, 'S') matches: False
  ValueCondition(NEGATION, 'S') matches: True
  ValueCondition(WILDCARD) matches: True
Node ID: 2, Text: 'andis', UPOS: V
  ValueCondition(EXACT, 'S') matches: False
  ValueCondition(NEGATION, 'S') matches: True
  ValueCondition(WILDCARD) matches: True
Node ID: 3, Text: 'lendurist', UPOS: S
  ValueCondition(EXACT, 'S') matches: True
  ValueCondition(NEGATION, 'S') matches: False
  ValueCondition(WILDCARD) matches: True
Node ID: 4, Text: 'abikaasale', UPOS: S
  ValueCondition(EXACT, 'S') matches: True
  ValueCondition(NEGATION, 'S') matches: False
  ValueCondition(WILDCARD) matches: True
Node ID: 5, Text: 'oma', UPOS: P
  ValueCondition(EXACT, 'S') matches: False
  ValueCondition(NEGATION, 'S') matches: True
  ValueCondition(WILDCARD) matches: True
Node ID: 6, Text: 'raamatu', UPOS: S
  ValueCondition(EXACT, 'S') matches: True
  ValueConditi

In [10]:
# FeatureCondition class testing
from typing import Type
from scripts.DepChainTagger import FeatureCondition, ConditionMode


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def run_featurecondition_tests() -> None:
    """Run behavioural and validation tests for FeatureCondition."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # 1) EXACT: required+forbidden positive case
    exact = FeatureCondition(
        mode=ConditionMode.EXACT,
        required={"Case": "Gen", "Number": "Sing"},
        forbidden={"Polarity": "Neg"},
        allow_extra_keys=False,
    )
    check(
        exact.matches({"Case": "Gen", "Number": "Sing", "Polarity": "Pos"}),
        "EXACT should pass when required keys match and forbidden keys do not.",
    )

    # 2) EXACT: required mismatch should fail
    check(
        not exact.matches({"Case": "Nom", "Number": "Sing", "Polarity": "Pos"}),
        "EXACT should fail when any required key has wrong value.",
    )

    # 3) EXACT: forbidden match should fail
    check(
        not exact.matches({"Case": "Gen", "Number": "Sing", "Polarity": "Neg"}),
        "EXACT should fail when forbidden key-value is present.",
    )

    # 4) EXACT: missing required key (allow_missing=False) should fail
    check(
        not exact.matches({"Case": "Gen", "Polarity": "Pos"}),
        "EXACT should fail when required key is missing and allow_missing=False.",
    )

    # 5) EXACT: missing required key (allow_missing=True) should pass
    exact_allow_missing = FeatureCondition(
        mode=ConditionMode.EXACT,
        required={"Case": "Gen", "Number": "Sing"},
        forbidden={"Polarity": "Neg"},
        allow_missing=True,
        allow_extra_keys=True,
    )
    check(
        exact_allow_missing.matches({"Case": "Gen"}),
        "EXACT should allow missing required keys when allow_missing=True.",
    )

    # 6) EXACT: extra-key policy
    exact_no_extra = FeatureCondition(
        mode=ConditionMode.EXACT,
        required={"Case": "Gen"},
        forbidden={"Polarity": "Neg"},
        allow_extra_keys=False,
    )
    check(
        not exact_no_extra.matches({"Case": "Gen", "Other": "X"}),
        "EXACT should reject unknown keys when allow_extra_keys=False.",
    )

    exact_with_extra = FeatureCondition(
        mode=ConditionMode.EXACT,
        required={"Case": "Gen"},
        forbidden={"Polarity": "Neg"},
        allow_extra_keys=True,
    )
    check(
        exact_with_extra.matches({"Case": "Gen", "Other": "X"}),
        "EXACT should allow unknown keys when allow_extra_keys=True.",
    )

    # 7) NEGATION: full required pattern match should fail
    neg = FeatureCondition(
        mode=ConditionMode.NEGATION,
        required={"Case": "Gen", "Number": "Sing"},
        forbidden={"Polarity": "Neg"},
    )
    check(
        not neg.matches({"Case": "Gen", "Number": "Sing"}),
        "NEGATION should fail when all required pairs match exactly.",
    )

    # 8) NEGATION: partial/non-match of required should pass
    check(
        neg.matches({"Case": "Gen", "Number": "Plur"}),
        "NEGATION should pass when full required pattern does not match.",
    )

    # 9) NEGATION: forbidden pair match should still fail
    check(
        not neg.matches({"Case": "Gen", "Number": "Plur", "Polarity": "Neg"}),
        "NEGATION should fail when forbidden pair is present.",
    )

    # 10) WILDCARD: should match any dict and also None
    wildcard = FeatureCondition(mode=ConditionMode.WILDCARD)
    check(wildcard.matches({"anything": "goes"}), "WILDCARD should match any dict.")
    check(wildcard.matches(None), "WILDCARD should match None as well.")

    # 11) Normalizer should normalise expected and actual values
    norm_cond = FeatureCondition(
        mode=ConditionMode.EXACT,
        required={"Case": "gEn"},
        forbidden={"Polarity": "nEg"},
        normalizer=lambda x: x.lower() if isinstance(x, str) else x,
        allow_extra_keys=True,
    )
    check(
        norm_cond.matches({"Case": "GEN", "Polarity": "pos"}),
        "Normalizer should make string comparison case-insensitive.",
    )

    # 12) Constructor validation checks
    assert_raises(
        ValueError,
        FeatureCondition,
        mode=ConditionMode.EXACT,
    )
    assert_raises(
        ValueError,
        FeatureCondition,
        mode=ConditionMode.WILDCARD,
        required={"Case": "Gen"},
    )
    assert_raises(
        TypeError,
        FeatureCondition,
        mode=ConditionMode.EXACT,
        required={"Case": "Gen"},
        normalizer="not-callable",
    )
    assert_raises(
        TypeError,
        FeatureCondition,
        mode=ConditionMode.EXACT,
        required=[("Case", "Gen")],
    )
    passed += 4
    total += 4

    print(f"FeatureCondition tests passed: {passed}/{total}")


run_featurecondition_tests()

FeatureCondition tests passed: 17/17


In [11]:
def parse_feat(feats_str: str) -> dict:
    """Parse a string into a dict."""
    feats = {}
    if feats_str.strip() == "_":
        return feats  # Return empty dict for underscore
    parts = feats_str.split()
    for i in range(0, len(parts)):
        key = parts[i]
        value = parts[i]
        feats[key] = value
    return feats

In [12]:
parse_feat("sg ill")  # Example usage

{'sg': 'sg', 'ill': 'ill'}

In [13]:
# Testing FeatureCondition on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the FeatureCondition implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing FeatureCondition on nodes:")
    for node in graph_index.iter_nodes():
        node_feats = node.feats if node.feats else {}
        # print(node)
        condition = FeatureCondition(
            mode=ConditionMode.EXACT,
            required={"sg": "sg", "n": "n"},
            # forbidden={"g": "g"},
            # allow_extra_keys=True,
            # allow_missing=True,
        )
        result = condition.matches(node_feats)
        print(f"Node ID: {node.id}, Text: '{node.text}', Feats: {node.feats}")
        print(f"  FeatureCondition(EXACT, required Case=Gen) matches: {result}")
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing FeatureCondition on nodes:
Node ID: 1, Text: 'Ta', Feats: OrderedDict([('sg', 'sg'), ('n', 'n')])
  FeatureCondition(EXACT, required Case=Gen) matches: True
Node ID: 2, Text: 'andis', Feats: OrderedDict([('s', 's')])
  FeatureCondition(EXACT, required Case=Gen) matches: False
Node ID: 3, Text: 'lendurist', Feats: OrderedDict([('sg', 'sg'), ('el', 'el')])
  FeatureCondition(EXACT, required Case=Gen) matches: False
Node ID: 4, Text: 'abikaasale', Feats: OrderedDict([('sg', 'sg'), ('all', 'all')])
  FeatureCondition(EXACT, required Case=Gen) matches: False
Node ID: 5, Text: 'oma', Feats: OrderedDict([('sg', 'sg'), ('g', 'g')])
  FeatureCondition(EXACT, required Case=Gen) matches: False
Node ID: 6, Text: 'raamatu', Feats: OrderedDict([('sg', 'sg'), ('g', 'g')])
  FeatureCondition(EXACT, required Case=Gen) matches: False
Node ID: 7, Text: '.', Feats: OrderedDict()
  FeatureCondition(EXACT, required Case=Gen) matches: False
---------------------

In [14]:
# NodeConstraint class testing
from types import SimpleNamespace
from typing import Type
from scripts.DepChainTagger import (
    NodeConstraint,
    ValueCondition,
    FeatureCondition,
    ConditionMode,
    NodePredicate,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def make_node(
    text: str = "test",
    upostag: str = "NOUN",
    xpostag: str = "S",
    lemma: str = "kass",
    deprel: str = "nmod",
    feats: dict | None = None,
) -> SimpleNamespace:
    """Create a lightweight span-like node object for NodeConstraint tests."""
    return SimpleNamespace(
        text=text,
        upostag=upostag,
        xpostag=xpostag,
        lemma=lemma,
        deprel=deprel,
        feats={} if feats is None else feats,
    )


def run_nodeconstraint_tests() -> None:
    """Run behavioural and validation tests for NodeConstraint."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # Base node for most checks
    node = make_node(
        upostag="NOUN",
        xpostag="S",
        lemma="lendur",
        deprel="nmod",
        feats={"sg": "sg", "n": "n"},
    )

    # 1) Happy path with multiple constraints
    constraint = NodeConstraint(
        role="target",
        upostag_condition=ValueCondition(ConditionMode.EXACT, "NOUN"),
        xpostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
        lemma_condition=ValueCondition(ConditionMode.EXACT, "lendur"),
        deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
        feats_condition=FeatureCondition(
            mode=ConditionMode.EXACT,
            required={"sg": "sg", "n": "n"},
            allow_extra_keys=True,
        ),
    )
    check(constraint.matches(node), "Node should satisfy all configured constraints.")

    # 2) Failing one scalar condition should fail the whole match
    wrong_node = make_node(upostag="VERB", xpostag="V", lemma="andma", deprel="root")
    check(
        not constraint.matches(wrong_node),
        "Mismatch in any configured scalar condition should fail.",
    )

    # 3) Feature mismatch should fail
    feature_mismatch_node = make_node(
        upostag="NOUN",
        xpostag="S",
        lemma="lendur",
        deprel="nmod",
        feats={"sg": "sg", "g": "g"},
    )
    check(
        not constraint.matches(feature_mismatch_node),
        "Feature mismatch should fail when feats_condition is configured.",
    )

    # 4) Extra predicates should participate in matching
    pred_ok: NodePredicate = lambda n: n.text.startswith("l")
    pred_fail: NodePredicate = lambda n: n.text.endswith("z")
    pred_constraint_ok = NodeConstraint(
        role="pred_test",
        extra_predicates=(pred_ok,),
    )
    pred_constraint_fail = NodeConstraint(
        role="pred_test",
        extra_predicates=(pred_fail,),
    )
    predicate_node = make_node(text="lendur")
    check(
        pred_constraint_ok.matches(predicate_node),
        "Matching extra predicate should allow match.",
    )
    check(
        not pred_constraint_fail.matches(predicate_node),
        "Failing extra predicate should reject match.",
    )

    # 5) Selectivity score should increase with more restrictive constraints
    unconstrained = NodeConstraint(role="a")
    exact_upos = NodeConstraint(
        role="b",
        upostag_condition=ValueCondition(ConditionMode.EXACT, "NOUN"),
    )
    exact_upos_plus_feats = NodeConstraint(
        role="c",
        upostag_condition=ValueCondition(ConditionMode.EXACT, "NOUN"),
        feats_condition=FeatureCondition(
            mode=ConditionMode.EXACT,
            required={"sg": "sg"},
            allow_extra_keys=True,
        ),
    )
    check(
        unconstrained.score_selectivity() < exact_upos.score_selectivity(),
        "Adding EXACT scalar condition should increase selectivity score.",
    )
    check(
        exact_upos.score_selectivity() < exact_upos_plus_feats.score_selectivity(),
        "Adding EXACT feats condition should further increase selectivity score.",
    )

    # 6) describe() should include role and configured condition info
    desc = constraint.describe()
    check("Role: target" in desc, "describe() should include role.")
    check(
        "UPOS:" in desc and "Feats:" in desc,
        "describe() should include configured parts.",
    )

    # 7) Constructor validation checks
    assert_raises(TypeError, NodeConstraint, role="")
    assert_raises(
        TypeError,
        NodeConstraint,
        role="bad_upos",
        upostag_condition="not-a-valuecondition",
    )
    assert_raises(
        TypeError,
        NodeConstraint,
        role="bad_feats",
        feats_condition="not-a-featurecondition",
    )
    assert_raises(
        TypeError,
        NodeConstraint,
        role="bad_preds",
        extra_predicates=[lambda n: True],
    )
    assert_raises(
        TypeError,
        NodeConstraint,
        role="bad_pred_member",
        extra_predicates=("not-callable",),
    )
    passed += 5
    total += 5

    print(f"NodeConstraint tests passed: {passed}/{total}")


run_nodeconstraint_tests()

NodeConstraint tests passed: 14/14


In [15]:
# Testing NodeConstraint on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the NodeConstraint implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing NodeConstraint on nodes:")
    for node in graph_index.iter_nodes():
        constraint = NodeConstraint(
            role="test_constraint",
            upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
            feats_condition=FeatureCondition(
                mode=ConditionMode.EXACT,
                required={"sg": "sg", "n": "n"},
                # allow_extra_keys=True,
            ),
        )
        result = constraint.matches(node)
        print(
            f"Node ID: {node.id}, Text: '{node.text}', UPOS: {node.upostag}, Feats: {node.feats}"
        )
        print(f"  NodeConstraint matches: {result}")
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing NodeConstraint on nodes:
Node ID: 1, Text: 'Ta', UPOS: P, Feats: OrderedDict([('sg', 'sg'), ('n', 'n')])
  NodeConstraint matches: False
Node ID: 2, Text: 'andis', UPOS: V, Feats: OrderedDict([('s', 's')])
  NodeConstraint matches: False
Node ID: 3, Text: 'lendurist', UPOS: S, Feats: OrderedDict([('sg', 'sg'), ('el', 'el')])
  NodeConstraint matches: False
Node ID: 4, Text: 'abikaasale', UPOS: S, Feats: OrderedDict([('sg', 'sg'), ('all', 'all')])
  NodeConstraint matches: False
Node ID: 5, Text: 'oma', UPOS: P, Feats: OrderedDict([('sg', 'sg'), ('g', 'g')])
  NodeConstraint matches: False
Node ID: 6, Text: 'raamatu', UPOS: S, Feats: OrderedDict([('sg', 'sg'), ('g', 'g')])
  NodeConstraint matches: False
Node ID: 7, Text: '.', UPOS: Z, Feats: OrderedDict()
  NodeConstraint matches: False
--------------------------------------------------
Sentence ID: 1, Sentence Span: (43, 70)
Testing NodeConstraint on nodes:
Node ID: 1, Text: 'See', UPOS: 

In [16]:
# EdgeConstraint class testing
from typing import Type
from scripts.DepChainTagger import (
    EdgeConstraint,
    EdgeContext,
    DirectionMode,
    ValueCondition,
    ConditionMode,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def make_edge_context(
    direction: DirectionMode,
    deprel: str | None,
    hops: int,
    crosses_sentence: bool,
) -> EdgeContext:
    """Build an EdgeContext instance for tests."""
    ctx = EdgeContext()
    ctx.direction = direction
    ctx.deprel = deprel
    ctx.hops = hops
    ctx.crosses_sentence = crosses_sentence
    return ctx


def run_edgeconstraint_tests() -> None:
    """Run behavioural and validation tests for EdgeConstraint."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # 1) Happy path: UP direction, matching deprel, hops in range
    c_up = EdgeConstraint(
        direction=DirectionMode.UP,
        deprel_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
        min_hops=1,
        max_hops=2,
        allow_crossing_sentence=False,
    )
    ctx_up_ok = make_edge_context(DirectionMode.UP, "nmod", 1, False)
    check(c_up.matches(ctx_up_ok), "Expected matching UP context to pass.")

    # 2) Direction mismatch should fail
    ctx_wrong_dir = make_edge_context(DirectionMode.DOWN, "nmod", 1, False)
    check(not c_up.matches(ctx_wrong_dir), "Direction mismatch should fail.")

    # 3) BOTH direction should accept UP and DOWN
    c_both = EdgeConstraint(
        direction=DirectionMode.BOTH,
        deprel_condition=ValueCondition(ConditionMode.EXACT, "obl"),
        min_hops=1,
        max_hops=3,
    )
    check(
        c_both.matches(make_edge_context(DirectionMode.UP, "obl", 2, False)),
        "BOTH direction should accept UP.",
    )
    check(
        c_both.matches(make_edge_context(DirectionMode.DOWN, "obl", 2, False)),
        "BOTH direction should accept DOWN.",
    )

    # 4) Deprel mismatch should fail
    check(
        not c_up.matches(make_edge_context(DirectionMode.UP, "obl", 1, False)),
        "Deprel mismatch should fail.",
    )

    # 5) Hop bounds should be enforced
    check(
        not c_up.matches(make_edge_context(DirectionMode.UP, "nmod", 0, False)),
        "Hops below min_hops should fail.",
    )
    check(
        not c_up.matches(make_edge_context(DirectionMode.UP, "nmod", 3, False)),
        "Hops above max_hops should fail.",
    )

    # 6) Sentence-boundary crossing policy
    check(
        not c_up.matches(make_edge_context(DirectionMode.UP, "nmod", 1, True)),
        "Cross-sentence edge should fail when not allowed.",
    )
    c_cross = EdgeConstraint(
        direction=DirectionMode.UP,
        deprel_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
        min_hops=1,
        max_hops=2,
        allow_crossing_sentence=True,
    )
    check(
        c_cross.matches(make_edge_context(DirectionMode.UP, "nmod", 1, True)),
        "Cross-sentence edge should pass when allowed.",
    )

    # 7) describe() should include key config parts
    desc = c_up.describe()
    check("Direction: up" in desc, "describe() should include direction.")
    check("Deprel:" in desc, "describe() should include deprel description.")
    check("Hops:" in desc, "describe() should include hop range.")

    # 8) Constructor validation checks
    assert_raises(TypeError, EdgeConstraint, direction="up")
    assert_raises(
        TypeError,
        EdgeConstraint,
        direction=DirectionMode.UP,
        deprel_condition="not-a-valuecondition",
    )
    assert_raises(
        ValueError,
        EdgeConstraint,
        direction=DirectionMode.UP,
        min_hops=-1,
    )
    assert_raises(
        ValueError,
        EdgeConstraint,
        direction=DirectionMode.UP,
        max_hops=-1,
    )
    assert_raises(
        ValueError,
        EdgeConstraint,
        direction=DirectionMode.UP,
        min_hops=3,
        max_hops=1,
    )
    assert_raises(
        TypeError,
        EdgeConstraint,
        direction=DirectionMode.UP,
        allow_crossing_sentence="no",
    )
    passed += 6
    total += 6

    print(f"EdgeConstraint tests passed: {passed}/{total}")


run_edgeconstraint_tests()

EdgeConstraint tests passed: 18/18


In [17]:
# Testing EdgeConstraint on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the EdgeConstraint implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing EdgeConstraint on edges:")
    for node, parent, direction in graph_index.iter_edges():
        edge_ctx = EdgeContext()
        edge_ctx.direction = direction
        edge_ctx.deprel = node.deprel if node else None
        edge_ctx.hops = 1  # In this simple test, we only have direct edges
        edge_ctx.crosses_sentence = False  # All edges are within the same sentence

        constraint = EdgeConstraint(
            direction=DirectionMode.UP,
            deprel_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
            min_hops=1,
            max_hops=2,
            allow_crossing_sentence=False,
        )
        result = constraint.matches(edge_ctx)
        parent_text = parent.text if parent else "None"
        parent_id = parent.id if parent else "None"
        print(
            f"Edge from Node ID: {node.id} ('{node.text}') to Parent ID: {parent_id} ('{parent_text}'), Direction: {direction}, Deprel: {edge_ctx.deprel}"
        )
        print(f"  EdgeConstraint matches: {result}")
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing EdgeConstraint on edges:
Edge from Node ID: 1 ('Ta') to Parent ID: 2 ('andis'), Direction: up, Deprel: nsubj
  EdgeConstraint matches: False
Edge from Node ID: 2 ('andis') to Parent ID: 1 ('Ta'), Direction: down, Deprel: root
  EdgeConstraint matches: False
Edge from Node ID: 3 ('lendurist') to Parent ID: 4 ('abikaasale'), Direction: up, Deprel: nmod
  EdgeConstraint matches: True
Edge from Node ID: 4 ('abikaasale') to Parent ID: 3 ('lendurist'), Direction: down, Deprel: obj
  EdgeConstraint matches: False
Edge from Node ID: 4 ('abikaasale') to Parent ID: 2 ('andis'), Direction: up, Deprel: obj
  EdgeConstraint matches: False
Edge from Node ID: 2 ('andis') to Parent ID: 4 ('abikaasale'), Direction: down, Deprel: root
  EdgeConstraint matches: False
Edge from Node ID: 5 ('oma') to Parent ID: 6 ('raamatu'), Direction: up, Deprel: nmod
  EdgeConstraint matches: True
Edge from Node ID: 6 ('raamatu') to Parent ID: 5 ('oma'), Direction: down, De

In [18]:
# PathPattern class testing
from typing import Type

from scripts.DepChainTagger import (
    ConditionMode,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    ValueCondition,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def run_pathpattern_tests() -> None:
    """Run behavioural and validation tests for PathPattern."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # 1) Happy path: two nodes connected by one edge.
    source_node = NodeConstraint(
        role="source",
        upostag_condition=ValueCondition(ConditionMode.EXACT, "NOUN"),
    )
    target_node = NodeConstraint(
        role="target",
        upostag_condition=ValueCondition(ConditionMode.EXACT, "ADJ"),
        deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "amod"),
    )
    source_to_target = EdgeConstraint(
        direction=DirectionMode.UP,
        deprel_condition=ValueCondition(ConditionMode.EXACT, "amod"),
        min_hops=1,
        max_hops=1,
        allow_crossing_sentence=False,
    )

    pattern = PathPattern(
        name="noun_to_adj",
        node_steps=(source_node, target_node),
        edge_steps=(source_to_target,),
        anchor_role="source",
        emit_roles=("source", "target"),
    )

    check(pattern.name == "noun_to_adj", "Pattern name should be preserved.")
    check(
        pattern.get_node_constraint("source") is source_node,
        "get_node_constraint should return the source node constraint.",
    )
    check(
        pattern.get_node_constraint("target") is target_node,
        "get_node_constraint should return the target node constraint.",
    )
    check(
        pattern.get_node_constraint("missing") is None,
        "get_node_constraint should return None for an unknown role.",
    )
    check(
        pattern.get_edge_constraint("source", "target") is source_to_target,
        "get_edge_constraint should return the matching edge constraint.",
    )
    check(
        pattern.get_edge_constraint("target", "source") is None,
        "get_edge_constraint should return None when the role order does not match.",
    )

    description = pattern.describe()
    check(
        "Pattern name: noun_to_adj" in description,
        "describe() should include the pattern name.",
    )
    check("Node step 0:" in description, "describe() should list node steps.")
    check("Edge step 0:" in description, "describe() should list edge steps.")
    check(
        "Anchor role: source" in description,
        "describe() should include the anchor role.",
    )
    check(
        "Emit roles: source, target" in description,
        "describe() should include emit roles.",
    )

    # 2) Validation: edge/node length mismatch should fail fast.
    assert_raises(
        ValueError,
        PathPattern,
        name="bad_length",
        node_steps=(source_node, target_node),
        edge_steps=(),
        anchor_role="source",
        emit_roles=("source",),
    )
    passed += 1
    total += 1

    # 3) Validation: duplicate node roles should fail fast.
    duplicate_role_node = NodeConstraint(role="source")
    assert_raises(
        ValueError,
        PathPattern,
        name="duplicate_roles",
        node_steps=(duplicate_role_node, duplicate_role_node),
        edge_steps=(source_to_target,),
        anchor_role="source",
        emit_roles=("source",),
    )
    passed += 1
    total += 1

    # 4) Validation: anchor role must exist in node_steps.
    assert_raises(
        ValueError,
        PathPattern,
        name="bad_anchor",
        node_steps=(source_node, target_node),
        edge_steps=(source_to_target,),
        anchor_role="not_present",
        emit_roles=("source",),
    )
    passed += 1
    total += 1

    # 5) Validation: emit roles must be a subset of node roles and unique.
    assert_raises(
        ValueError,
        PathPattern,
        name="bad_emit_subset",
        node_steps=(source_node, target_node),
        edge_steps=(source_to_target,),
        anchor_role="source",
        emit_roles=("source", "other"),
    )
    passed += 1
    total += 1

    assert_raises(
        ValueError,
        PathPattern,
        name="bad_emit_duplicates",
        node_steps=(source_node, target_node),
        edge_steps=(source_to_target,),
        anchor_role="source",
        emit_roles=("source", "source"),
    )
    passed += 1
    total += 1

    # 6) Constructor validation: basic type checks.
    assert_raises(
        TypeError,
        PathPattern,
        name="",
        node_steps=(source_node,),
        edge_steps=(),
        anchor_role="source",
        emit_roles=("source",),
    )
    assert_raises(
        TypeError,
        PathPattern,
        name="bad_nodes",
        node_steps=[source_node, target_node],
        edge_steps=(source_to_target,),
        anchor_role="source",
        emit_roles=("source",),
    )
    assert_raises(
        TypeError,
        PathPattern,
        name="bad_edges",
        node_steps=(source_node, target_node),
        edge_steps=[source_to_target],
        anchor_role="source",
        emit_roles=("source",),
    )
    assert_raises(
        TypeError,
        PathPattern,
        name="bad_anchor_type",
        node_steps=(source_node, target_node),
        edge_steps=(source_to_target,),
        anchor_role=123,
        emit_roles=("source",),
    )
    assert_raises(
        TypeError,
        PathPattern,
        name="bad_emit_type",
        node_steps=(source_node, target_node),
        edge_steps=(source_to_target,),
        anchor_role="source",
        emit_roles=["source"],
    )
    passed += 5
    total += 5

    print(f"PathPattern tests passed: {passed}/{total}")


run_pathpattern_tests()

PathPattern tests passed: 21/21


In [19]:
# Testing PathPattern on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the PathPattern implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing PathPattern on node pairs:")
    for node in graph_index.iter_nodes():
        for other_node in graph_index.iter_nodes():
            if node.id >= other_node.id:
                continue  # Avoid duplicate pairs and self-pairing

            # Define a simple pattern that looks for a NOUN connected to an ADJ via an amod relation
            source_node = NodeConstraint(
                role="source",
                upostag_condition=ValueCondition(ConditionMode.EXACT, "NOUN"),
            )
            target_node = NodeConstraint(
                role="target",
                upostag_condition=ValueCondition(ConditionMode.EXACT, "ADJ"),
                deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "amod"),
            )
            edge_constraint = EdgeConstraint(
                direction=DirectionMode.UP,
                deprel_condition=ValueCondition(ConditionMode.EXACT, "amod"),
                min_hops=1,
                max_hops=1,
                allow_crossing_sentence=False,
            )
            pattern = PathPattern(
                name="noun_to_adj",
                node_steps=(source_node, target_node),
                edge_steps=(edge_constraint,),
                anchor_role="source",
                emit_roles=("source", "target"),
            )

            # In a real test, we would have a method like pattern.matches(graph_index, node_pair)
            # that checks if the pattern matches between node and other_node. Here we just print the pair.
            print(
                f"Node pair: ({node.id}, '{node.text}') -> ({other_node.id}, '{other_node.text}')"
            )
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing PathPattern on node pairs:
Node pair: (1, 'Ta') -> (2, 'andis')
Node pair: (1, 'Ta') -> (3, 'lendurist')
Node pair: (1, 'Ta') -> (4, 'abikaasale')
Node pair: (1, 'Ta') -> (5, 'oma')
Node pair: (1, 'Ta') -> (6, 'raamatu')
Node pair: (1, 'Ta') -> (7, '.')
Node pair: (2, 'andis') -> (3, 'lendurist')
Node pair: (2, 'andis') -> (4, 'abikaasale')
Node pair: (2, 'andis') -> (5, 'oma')
Node pair: (2, 'andis') -> (6, 'raamatu')
Node pair: (2, 'andis') -> (7, '.')
Node pair: (3, 'lendurist') -> (4, 'abikaasale')
Node pair: (3, 'lendurist') -> (5, 'oma')
Node pair: (3, 'lendurist') -> (6, 'raamatu')
Node pair: (3, 'lendurist') -> (7, '.')
Node pair: (4, 'abikaasale') -> (5, 'oma')
Node pair: (4, 'abikaasale') -> (6, 'raamatu')
Node pair: (4, 'abikaasale') -> (7, '.')
Node pair: (5, 'oma') -> (6, 'raamatu')
Node pair: (5, 'oma') -> (7, '.')
Node pair: (6, 'raamatu') -> (7, '.')
--------------------------------------------------
Sentence ID: 1, Sentenc

In [20]:
# ChainMatch class testing
from typing import Type

from scripts.DepChainTagger import (
    ChainMatch,
    DirectionMode,
    EdgeContext,
    SyntaxGraphIndex,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def make_edge_context(
    direction: DirectionMode,
    deprel: str | None,
    hops: int,
    crosses_sentence: bool,
) -> EdgeContext:
    """Build an EdgeContext instance for ChainMatch tests."""
    ctx = EdgeContext()
    ctx.direction = direction
    ctx.deprel = deprel
    ctx.hops = hops
    ctx.crosses_sentence = crosses_sentence
    return ctx


def run_chainmatch_tests() -> None:
    """Run behavioural and validation tests for ChainMatch."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # Build real estnltk.Span nodes from a sentence to satisfy strict type validation.
    sample_text = "Ta andis lendurist abikaasale oma raamatu."
    text_obj = estnltk.Text(sample_text)
    text_obj.tag_layer("morph_extended")
    stanza_syntax_tagger.tag(text_obj)

    sentence = text_obj.sentences[0]
    graph = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=0,
        sentence_span=(sentence.start, sentence.end),
    )

    source_node = graph.get_node(3)  # lendurist
    target_node = graph.get_node(4)  # abikaasale

    edge_ctx = make_edge_context(
        direction=DirectionMode.UP,
        deprel="nmod",
        hops=1,
        crosses_sentence=False,
    )

    match = ChainMatch(
        pattern_name="lendurist_to_abikaasale",
        sentence_index=0,
        sentence_span=(sentence.start, sentence.end),
        role_to_token_id={"source": 3, "target": 4},
        role_to_node={"source": source_node, "target": target_node},
        traversed_edges=(("source", "target", edge_ctx),),
        matched_text=f"{source_node.text} {target_node.text}",
        metadata={"confidence": 1.0},
    )

    # 1) Happy-path accessors and helpers.
    check(
        match.get_node("source").text == source_node.text,
        "get_node should return the correct node.",
    )
    check(
        match.get_token_id("target") == 4,
        "get_token_id should return the correct token id.",
    )
    check(
        match.get_roles() == {"source", "target"},
        "get_roles should return all matched roles.",
    )
    check(
        match.build_matched_text(("source", "target"))
        == f"{source_node.text} {target_node.text}",
        "build_matched_text should concatenate node texts in role order.",
    )

    # 2) to_output_row should expose key fields and edge metadata.
    row = match.to_output_row()
    check(
        row["pattern_name"] == "lendurist_to_abikaasale",
        "to_output_row should include pattern_name.",
    )
    check(
        row["role_to_token_id"]["source"] == 3,
        "to_output_row should include role_to_token_id mapping.",
    )
    check(
        row["traversed_edges"][0]["from_role"] == "source",
        "to_output_row should include traversed edge roles.",
    )
    check(
        row["traversed_edges"][0]["edge_context"]["direction"] == "up",
        "to_output_row should serialise edge direction.",
    )

    # 3) describe() should include key summary text.
    desc = match.describe()
    check(
        "Pattern name: lendurist_to_abikaasale" in desc,
        "describe() should include pattern name.",
    )
    check("Matched text:" in desc, "describe() should include matched text.")

    # 4) Missing role lookups should raise ValueError.
    assert_raises(ValueError, match.get_node, "missing")
    assert_raises(ValueError, match.get_token_id, "missing")
    passed += 2
    total += 2

    # 5) Constructor validation checks.
    assert_raises(
        ValueError,
        ChainMatch,
        pattern_name="x",
        sentence_index=0,
        sentence_span=(10, 0),
        role_to_token_id={"source": 3},
        role_to_node={"source": source_node},
        traversed_edges=(("source", "source", edge_ctx),),
        matched_text="x",
    )

    assert_raises(
        ValueError,
        ChainMatch,
        pattern_name="x",
        sentence_index=0,
        sentence_span=(0, 10),
        role_to_token_id={"source": 3},
        role_to_node={"target": target_node},
        traversed_edges=(("source", "target", edge_ctx),),
        matched_text="x",
    )

    assert_raises(
        TypeError,
        ChainMatch,
        pattern_name="x",
        sentence_index=0,
        sentence_span=(0, 10),
        role_to_token_id={"source": 3},
        role_to_node={"source": "not-a-span"},
        traversed_edges=(("source", "source", edge_ctx),),
        matched_text="x",
    )

    assert_raises(
        TypeError,
        ChainMatch,
        pattern_name="x",
        sentence_index=0,
        sentence_span=(0, 10),
        role_to_token_id={"source": 3},
        role_to_node={"source": source_node},
        traversed_edges=(("source",),),
        matched_text="x",
    )

    assert_raises(
        TypeError,
        ChainMatch,
        pattern_name="x",
        sentence_index=0,
        sentence_span=(0, 10),
        role_to_token_id={"source": 3},
        role_to_node={"source": source_node},
        traversed_edges=(("source", "source", edge_ctx),),
        matched_text="x",
        metadata="not-a-dict",
    )
    passed += 5
    total += 5

    print(f"ChainMatch tests passed: {passed}/{total}")


run_chainmatch_tests()

ChainMatch tests passed: 17/17


In [21]:
# Testing ChainMatch on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the ChainMatch implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing ChainMatch on a sample match:")
    source_node = graph_index.get_node(3)  # lendurist
    target_node = graph_index.get_node(4)  # abikaasale
    edge_ctx = EdgeContext()
    edge_ctx.direction = DirectionMode.UP
    edge_ctx.deprel = "nmod"
    edge_ctx.hops = 1
    edge_ctx.crosses_sentence = False

    match = ChainMatch(
        pattern_name="lendurist_to_abikaasale",
        sentence_index=id,
        sentence_span=(sentence.start, sentence.end),
        role_to_token_id={"source": 3, "target": 4},
        role_to_node={"source": source_node, "target": target_node},
        traversed_edges=(("source", "target", edge_ctx),),
        matched_text=f"{source_node.text} {target_node.text}",
    )

    print(match.describe())
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing ChainMatch on a sample match:
Pattern name: lendurist_to_abikaasale
Sentence index: 0
Sentence span: (0, 42)
Matched roles:
  Role: source, Token ID: 3, Node properties: {start: 9, end: 18, upostag: S, xpostag: S, lemma: lendur, feats: OrderedDict([('sg', 'sg'), ('el', 'el')])}
  Role: target, Token ID: 4, Node properties: {start: 19, end: 29, upostag: S, xpostag: S, lemma: abikaasa, feats: OrderedDict([('sg', 'sg'), ('all', 'all')])}
Traversed edges:
  From role: source to role: target, Edge context: {direction: up, deprel: nmod, hops: 1, crosses_sentence: False}
Matched text: 'lendurist abikaasale'
--------------------------------------------------
Sentence ID: 1, Sentence Span: (43, 70)
Testing ChainMatch on a sample match:
Pattern name: lendurist_to_abikaasale
Sentence index: 1
Sentence span: (43, 70)
Matched roles:
  Role: source, Token ID: 3, Node properties: {start: 54, end: 56, upostag: V, xpostag: V, lemma: olema, feats: OrderedDi

In [22]:
# MatchCollector class testing
from typing import Type

from scripts.DepChainTagger import (
    ChainMatch,
    DirectionMode,
    EdgeContext,
    MatchCollector,
    SyntaxGraphIndex,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def make_edge_context(
    direction: DirectionMode,
    deprel: str | None,
    hops: int,
    crosses_sentence: bool,
) -> EdgeContext:
    """Build an EdgeContext instance for MatchCollector tests."""
    ctx = EdgeContext()
    ctx.direction = direction
    ctx.deprel = deprel
    ctx.hops = hops
    ctx.crosses_sentence = crosses_sentence
    return ctx


def make_chain_match(
    pattern_name: str,
    sentence_index: int,
    sentence_span: tuple[int, int],
    source_id: int,
    target_id: int,
    source_node,
    target_node,
    matched_text: str,
    metadata: dict | None = None,
) -> ChainMatch:
    """Create a ChainMatch for testing collector behaviour."""
    edge_ctx = make_edge_context(
        direction=DirectionMode.UP,
        deprel="nmod",
        hops=1,
        crosses_sentence=False,
    )
    return ChainMatch(
        pattern_name=pattern_name,
        sentence_index=sentence_index,
        sentence_span=sentence_span,
        role_to_token_id={"source": source_id, "target": target_id},
        role_to_node={"source": source_node, "target": target_node},
        traversed_edges=(("source", "target", edge_ctx),),
        matched_text=matched_text,
        metadata={} if metadata is None else metadata,
    )


def run_matchcollector_tests() -> None:
    """Run behavioural and validation tests for MatchCollector."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # Build real estnltk.Span nodes via SyntaxGraphIndex for valid ChainMatch objects.
    sample_text = "Ta andis lendurist abikaasale oma raamatu."
    text_obj = estnltk.Text(sample_text)
    text_obj.tag_layer("morph_extended")
    stanza_syntax_tagger.tag(text_obj)

    sentence = text_obj.sentences[0]
    graph = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=0,
        sentence_span=(sentence.start, sentence.end),
    )
    node_3 = graph.get_node(3)  # lendurist
    node_4 = graph.get_node(4)  # abikaasale
    node_5 = graph.get_node(5)  # oma
    node_6 = graph.get_node(6)  # raamatu

    match_a = make_chain_match(
        pattern_name="p1",
        sentence_index=0,
        sentence_span=(sentence.start, sentence.end),
        source_id=3,
        target_id=4,
        source_node=node_3,
        target_node=node_4,
        matched_text=f"{node_3.text} {node_4.text}",
        metadata={"m": 1},
    )
    match_a_variant = make_chain_match(
        pattern_name="p1",
        sentence_index=0,
        sentence_span=(sentence.start, sentence.end),
        source_id=3,
        target_id=4,
        source_node=node_3,
        target_node=node_4,
        matched_text="different text",
        metadata={"m": 999},
    )
    match_b = make_chain_match(
        pattern_name="p2",
        sentence_index=0,
        sentence_span=(sentence.start, sentence.end),
        source_id=5,
        target_id=6,
        source_node=node_5,
        target_node=node_6,
        matched_text=f"{node_5.text} {node_6.text}",
        metadata={"m": 2},
    )

    # 1) dedup_mode='none' should accept duplicates and count correctly.
    collector_none = MatchCollector(dedup_mode="none", max_matches=10)
    check(collector_none.add(match_a), "First add should succeed in dedup_mode='none'.")
    check(
        collector_none.add(match_a),
        "Duplicate add should also succeed in dedup_mode='none'.",
    )
    check(
        collector_none.count() == 2,
        "Collector count should include duplicates in dedup_mode='none'.",
    )

    # 2) dedup_mode='exact' should reject exact duplicates.
    collector_exact = MatchCollector(dedup_mode="exact", max_matches=10)
    check(
        collector_exact.add(match_a), "First add should succeed in dedup_mode='exact'."
    )
    check(
        not collector_exact.add(match_a),
        "Exact duplicate should be rejected in dedup_mode='exact'.",
    )
    check(
        collector_exact.count() == 1,
        "Collector count should remain 1 after rejecting exact duplicate.",
    )

    # 3) dedup_mode='role_based' should reject same role assignments even if metadata differs.
    collector_role = MatchCollector(dedup_mode="role_based", max_matches=10)
    check(
        collector_role.add(match_a),
        "First add should succeed in dedup_mode='role_based'.",
    )
    check(
        not collector_role.add(match_a_variant),
        "Role-based duplicate should be rejected even when non-key fields differ.",
    )
    check(
        collector_role.add(match_b),
        "Different role mapping should be accepted in dedup_mode='role_based'.",
    )
    check(
        collector_role.count() == 2,
        "Collector count should be 2 for two unique role-based matches.",
    )

    # 4) extend() should return number of accepted matches.
    collector_extend = MatchCollector(dedup_mode="role_based", max_matches=10)
    added = collector_extend.extend([match_a, match_a_variant, match_b])
    check(added == 2, "extend() should report only accepted matches.")

    # 5) max_matches should cap storage.
    collector_capped = MatchCollector(dedup_mode="none", max_matches=1)
    check(collector_capped.add(match_a), "First add should succeed under cap.")
    check(
        not collector_capped.add(match_b), "Add beyond max_matches should be rejected."
    )

    # 6) summary(), all(), to_output_rows(), clear() behaviour.
    summary = collector_role.summary()
    check(summary["total"] == 2, "summary() should include total count.")
    check(summary["pattern::p1"] == 1, "summary() should include per-pattern counts.")
    check(
        summary["pattern::p2"] == 1,
        "summary() should include per-pattern counts for each pattern.",
    )

    rows = collector_role.to_output_rows()
    check(len(rows) == 2, "to_output_rows() should return one row per stored match.")
    check(
        rows[0]["pattern_name"] in {"p1", "p2"},
        "to_output_rows() should preserve match fields.",
    )

    all_matches = collector_role.all()
    check(len(all_matches) == 2, "all() should return all stored matches.")

    collector_role.clear()
    check(collector_role.count() == 0, "clear() should remove all stored matches.")

    # 7) Constructor validation checks.
    assert_raises(ValueError, MatchCollector, dedup_mode="invalid")
    assert_raises(ValueError, MatchCollector, dedup_mode="none", max_matches=0)
    assert_raises(TypeError, MatchCollector, matches="not-a-list")
    assert_raises(TypeError, MatchCollector, matches=["not-a-chainmatch"])
    passed += 4
    total += 4

    print(f"MatchCollector tests passed: {passed}/{total}")


run_matchcollector_tests()

MatchCollector tests passed: 24/24


In [23]:
# Testing MatchCollector on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the MatchCollector implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing MatchCollector with sample matches:")
    collector = MatchCollector(dedup_mode="role_based", max_matches=10)

    node_3 = graph_index.get_node(3)  # lendurist
    node_4 = graph_index.get_node(4)  # abikaasale
    node_5 = graph_index.get_node(5)  # oma
    node_6 = graph_index.get_node(6)  # raamatu

    match_a = make_chain_match(
        pattern_name="p1",
        sentence_index=id,
        sentence_span=(sentence.start, sentence.end),
        source_id=3,
        target_id=4,
        source_node=node_3,
        target_node=node_4,
        matched_text=f"{node_3.text} {node_4.text}",
        metadata={"m": 1},
    )
    match_b = make_chain_match(
        pattern_name="p2",
        sentence_index=id,
        sentence_span=(sentence.start, sentence.end),
        source_id=5,
        target_id=6,
        source_node=node_5,
        target_node=node_6,
        matched_text=f"{node_5.text} {node_6.text}",
        metadata={"m": 2},
    )

    collector.add(match_a)
    collector.add(match_b)

    print("Match summary:", collector.summary())
    for row in collector.to_output_rows():
        print("Output row:", row)
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing MatchCollector with sample matches:
Match summary: {'total': 2, 'pattern::p1': 1, 'pattern::p2': 1}
Output row: {'pattern_name': 'p1', 'sentence_index': 0, 'sentence_span': (0, 42), 'role_to_token_id': {'source': 3, 'target': 4}, 'role_to_node': {'source': {'start': 9, 'end': 18, 'upostag': 'S', 'xpostag': 'S', 'lemma': 'lendur', 'feats': OrderedDict([('sg', 'sg'), ('el', 'el')])}, 'target': {'start': 19, 'end': 29, 'upostag': 'S', 'xpostag': 'S', 'lemma': 'abikaasa', 'feats': OrderedDict([('sg', 'sg'), ('all', 'all')])}}, 'traversed_edges': [{'from_role': 'source', 'to_role': 'target', 'edge_context': {'direction': 'up', 'deprel': 'nmod', 'hops': 1, 'crosses_sentence': False}}], 'matched_text': 'lendurist abikaasale', 'metadata': {'m': 1}}
Output row: {'pattern_name': 'p2', 'sentence_index': 0, 'sentence_span': (0, 42), 'role_to_token_id': {'source': 5, 'target': 6}, 'role_to_node': {'source': {'start': 30, 'end': 33, 'upostag': 'P', 'xpo

In [24]:
# DepChainMatcher class testing
from typing import Type

from scripts.DepChainTagger import (
    ConditionMode,
    DepChainMatcher,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    SyntaxGraphIndex,
    ValueCondition,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def build_simple_pattern(name: str) -> PathPattern:
    """Build a pattern matching any parent-child relationship."""
    parent_node = NodeConstraint(
        role="parent",
        upostag_condition=ValueCondition(ConditionMode.WILDCARD),
    )
    child_node = NodeConstraint(
        role="child",
        upostag_condition=ValueCondition(ConditionMode.WILDCARD),
    )
    edge_constraint = EdgeConstraint(
        direction=DirectionMode.UP,
        deprel_condition=ValueCondition(ConditionMode.WILDCARD),
        min_hops=1,
        max_hops=1,
        allow_crossing_sentence=False,
    )
    return PathPattern(
        name=name,
        node_steps=(parent_node, child_node),
        edge_steps=(edge_constraint,),
        anchor_role="child",
        emit_roles=("parent", "child"),
    )


def run_depchainmatcher_tests() -> None:
    """Run behavioural and validation tests for DepChainMatcher."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    sample_text = "Ta andis lendurist abikaasale oma raamatu."
    text_obj = estnltk.Text(sample_text)
    text_obj.tag_layer("morph_extended")
    stanza_syntax_tagger.tag(text_obj)

    sentence = text_obj.sentences[0]
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=0,
        sentence_span=(sentence.start, sentence.end),
    )

    base_pattern = build_simple_pattern("base_p")

    # 1) match_pattern_in_sentence should produce matches with role-token mapping.
    matcher = DepChainMatcher(patterns=(base_pattern,))
    pattern_matches = matcher.match_pattern_in_sentence(
        pattern=base_pattern,
        graph_index=graph_index,
        sentence_index=0,
    )
    check(len(pattern_matches) >= 1, "Expected at least one match for base pattern.")

    first_match = pattern_matches[0]
    check(
        "parent" in first_match.role_to_token_id
        and "child" in first_match.role_to_token_id,
        "Match should have both parent and child roles mapped.",
    )

    # 2) sentence_span override should be respected.
    overridden = matcher.match_pattern_in_sentence(
        pattern=base_pattern,
        graph_index=graph_index,
        sentence_index=0,
        sentence_span=(999, 1001),
    )
    check(len(overridden) >= 1, "Override run should still produce at least one match.")
    check(
        overridden[0].sentence_span == (999, 1001),
        "Explicit sentence_span should override graph sentence span.",
    )

    # 3) match_sentence should aggregate all configured patterns.
    second_pattern = build_simple_pattern("base_p_2")
    matcher_multi = DepChainMatcher(
        patterns=(base_pattern, second_pattern), dedup_mode="none"
    )
    multi_matches = matcher_multi.match_sentence(
        graph_index=graph_index, sentence_index=0
    )
    check(
        len(multi_matches) >= len(pattern_matches),
        "Aggregating multiple patterns should produce at least as many matches as one pattern.",
    )

    # 4) dedup_mode should control duplicate handling across patterns.
    # Compare 'none' mode (allows duplicates) vs 'role_based' (deduplicates by role assignment).
    matcher_none = DepChainMatcher(
        patterns=(base_pattern, base_pattern),
        dedup_mode="none",
    )
    none_matches = matcher_none.match_sentence(
        graph_index=graph_index, sentence_index=0
    )

    matcher_dedup = DepChainMatcher(
        patterns=(base_pattern, base_pattern),
        dedup_mode="role_based",
    )
    dedup_matches = matcher_dedup.match_sentence(
        graph_index=graph_index, sentence_index=0
    )

    # role_based dedup should produce same or fewer matches than 'none' mode
    # when same pattern is run twice.
    check(
        len(dedup_matches) <= len(none_matches),
        "role_based dedup should produce same or fewer matches than 'none' mode.",
    )

    # 5) allow_role_node_overlap should control whether same token can fill multiple roles.
    # Build a pattern where both roles match the same node (zero-hop self-match).
    same_node_pattern = PathPattern(
        name="self_ref",
        node_steps=(
            NodeConstraint(
                role="a", upostag_condition=ValueCondition(ConditionMode.WILDCARD)
            ),
            NodeConstraint(
                role="b", upostag_condition=ValueCondition(ConditionMode.WILDCARD)
            ),
        ),
        edge_steps=(
            EdgeConstraint(
                direction=DirectionMode.UP,
                deprel_condition=ValueCondition(ConditionMode.WILDCARD),
                min_hops=0,
                max_hops=0,
                allow_crossing_sentence=False,
            ),
        ),
        anchor_role="a",
        emit_roles=("a", "b"),
    )

    matcher_no_overlap = DepChainMatcher(
        patterns=(same_node_pattern,),
        allow_role_node_overlap=False,
    )
    no_overlap_matches = matcher_no_overlap.match_sentence(
        graph_index=graph_index,
        sentence_index=0,
    )

    matcher_with_overlap = DepChainMatcher(
        patterns=(same_node_pattern,),
        allow_role_node_overlap=True,
    )
    with_overlap_matches = matcher_with_overlap.match_sentence(
        graph_index=graph_index,
        sentence_index=0,
    )

    # With overlap disabled, only zero-hop matches to different nodes should occur.
    # With overlap enabled, zero-hop self-matches should also occur.
    check(
        len(with_overlap_matches) >= len(no_overlap_matches),
        "Overlap-enabled matcher should allow at least as many matches as overlap-disabled.",
    )

    # 6) Constructor validation checks.
    assert_raises(TypeError, DepChainMatcher, patterns=[base_pattern])
    assert_raises(
        ValueError, DepChainMatcher, patterns=(base_pattern,), dedup_mode="invalid"
    )
    assert_raises(
        ValueError,
        DepChainMatcher,
        patterns=(base_pattern,),
        max_matches_per_sentence=0,
    )
    assert_raises(
        TypeError,
        DepChainMatcher,
        patterns=(base_pattern,),
        allow_role_node_overlap="no",
    )
    passed += 4
    total += 4

    print(f"DepChainMatcher tests passed: {passed}/{total}")


run_depchainmatcher_tests()

DepChainMatcher tests passed: 11/11


In [25]:
print(text_obj.stanza_syntax)

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc', 'parent_span', 'children'), spans=SL[Span('Ta', [{'id': 1, 'lemma': 'tema', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('n', 'n')]), 'head': 2, 'deprel': 'nsubj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('andis', [{'id': 2, 'lemma': 'andma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('s', 's')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_', 'parent_span': None, 'children': <class 'tuple'>}]),
Span('lendurist', [{'id': 3, 'lemma': 'lendur', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('el', 'el')]), 'head': 4, 'deprel': 'nmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('abikaasale', [{'id': 4, 'lemma': 'abikaasa', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 

In [26]:
# Testing DepChainMatcher on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the DepChainMatcher implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}"
    )
    print("Testing DepChainMatcher with sample patterns:")
    pattern = PathPattern(
        name="noun_to_adj",
        node_steps=(
            NodeConstraint(
                role="parent",
                upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
            ),
            NodeConstraint(
                role="child",
                upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
                deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "obj"),
            ),
        ),
        edge_steps=(
            EdgeConstraint(
                direction=DirectionMode.UP,
                deprel_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
                min_hops=1,
                max_hops=1,
                allow_crossing_sentence=False,
            ),
        ),
        anchor_role="child",
        emit_roles=("parent", "child"),
    )

    matcher = DepChainMatcher(patterns=(pattern,), dedup_mode="none")
    matches = matcher.match_sentence(graph_index=graph_index, sentence_index=id)
    print(f"Found {len(matches)} matches for pattern '{pattern.name}'.")
    for match in matches:
        print(match.describe())
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42)
Testing DepChainMatcher with sample patterns:
Found 1 matches for pattern 'noun_to_adj'.
Pattern name: noun_to_adj
Sentence index: 0
Sentence span: (0, 42)
Matched roles:
  Role: child, Token ID: 4, Node properties: {start: 19, end: 29, upostag: S, xpostag: S, lemma: abikaasa, feats: OrderedDict([('sg', 'sg'), ('all', 'all')])}
  Role: parent, Token ID: 3, Node properties: {start: 9, end: 18, upostag: S, xpostag: S, lemma: lendur, feats: OrderedDict([('sg', 'sg'), ('el', 'el')])}
Traversed edges:
  From role: parent to role: child, Edge context: {direction: up, deprel: nmod, hops: 1, crosses_sentence: False}
Matched text: 'lendurist abikaasale'
--------------------------------------------------
Sentence ID: 1, Sentence Span: (43, 70)
Testing DepChainMatcher with sample patterns:
Found 0 matches for pattern 'noun_to_adj'.
--------------------------------------------------


In [27]:
# DepChainTaggerOrchestrator class testing
from typing import Type

from scripts.DepChainTagger import (
    ConditionMode,
    DepChainMatcher,
    DepChainTaggerOrchestrator,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    PhraseDecorator,
    SyntaxGraphIndex,
    ValueCondition,
)


def assert_raises(exc_type: Type[Exception], fn, *args, **kwargs) -> None:
    """Assert that calling fn(*args, **kwargs) raises the expected exception."""
    try:
        fn(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised.")


def build_wildcard_pattern(name: str) -> PathPattern:
    """Build a pattern matching any parent-child relationship."""
    parent_node = NodeConstraint(
        role="parent",
        upostag_condition=ValueCondition(ConditionMode.WILDCARD),
    )
    child_node = NodeConstraint(
        role="child",
        upostag_condition=ValueCondition(ConditionMode.WILDCARD),
    )
    edge_constraint = EdgeConstraint(
        direction=DirectionMode.UP,
        deprel_condition=ValueCondition(ConditionMode.WILDCARD),
        min_hops=1,
        max_hops=1,
        allow_crossing_sentence=False,
    )
    return PathPattern(
        name=name,
        node_steps=(parent_node, child_node),
        edge_steps=(edge_constraint,),
        anchor_role="child",
        emit_roles=("parent", "child"),
    )


def run_depchaintaggerorchestrator_tests() -> None:
    """Run behavioural and validation tests for DepChainTaggerOrchestrator."""
    passed = 0
    total = 0

    def check(condition: bool, message: str) -> None:
        """Validate one condition and fail with a readable message."""
        nonlocal passed, total
        total += 1
        if not condition:
            raise AssertionError(message)
        passed += 1

    # Prepare sample text with two sentences
    sample_text = (
        "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
    )
    text_obj = estnltk.Text(sample_text)
    text_obj.tag_layer("morph_extended")
    stanza_syntax_tagger.tag(text_obj)

    base_pattern = build_wildcard_pattern("base_p")

    # 1) Constructor should set defaults when matcher and decorator are not provided.
    orchestrator = DepChainTaggerOrchestrator(
        patterns=(base_pattern,),
        sentence_match_dedup_mode="role_based",
        max_matches_per_sentence=100,
        allow_role_node_overlap=False,
    )
    check(
        orchestrator.matcher is not None,
        "Matcher should be auto-created if not provided.",
    )
    check(
        orchestrator.decorator is not None,
        "Decorator should be auto-created if not provided.",
    )
    check(
        isinstance(orchestrator.matcher, DepChainMatcher),
        "Auto-created matcher should be DepChainMatcher instance.",
    )
    check(
        isinstance(orchestrator.decorator, PhraseDecorator),
        "Auto-created decorator should be PhraseDecorator instance.",
    )

    # 2) tag_sentence_layer() should process a single sentence and return matches.
    sentence_layer = text_obj.sentences[0].stanza_syntax
    matches = orchestrator.tag_sentence_layer(
        sentence_syntax_layer=sentence_layer,
        sentence_index=0,
        sentence_span=(0, 42),
    )
    check(
        isinstance(matches, list) and all(isinstance(m, ChainMatch) for m in matches),
        "tag_sentence_layer() should return a list of ChainMatch objects.",
    )

    # 3) tag_sentence_layers() should aggregate matches from multiple sentences.
    sentence_layers = [sent.stanza_syntax for sent in text_obj.sentences]
    sentence_spans = [(sent.start, sent.end) for sent in text_obj.sentences]

    all_matches = orchestrator.tag_sentence_layers(
        sentence_syntax_layers=sentence_layers,
        sentence_spans=sentence_spans,
    )
    check(
        isinstance(all_matches, list),
        "tag_sentence_layers() should return a list of matches.",
    )

    # 4) decorate_matches() should convert ChainMatch objects to output rows.
    if matches:  # Only test if we have matches
        decorated = orchestrator.decorate_matches(matches)
        check(
            isinstance(decorated, list)
            and all(isinstance(row, dict) for row in decorated),
            "decorate_matches() should return a list of dictionaries.",
        )
        if decorated:
            row = decorated[0]
            check(
                "matched_text" in row,
                "Decorated row should include 'matched_text' field.",
            )

    # 5) tag_and_decorate_sentence_layers() should run full pipeline.
    decorated_all = orchestrator.tag_and_decorate_sentence_layers(
        sentence_syntax_layers=sentence_layers,
        sentence_spans=sentence_spans,
    )
    check(
        isinstance(decorated_all, list),
        "tag_and_decorate_sentence_layers() should return decorated rows.",
    )

    # 6) global_dedup_mode should control cross-sentence deduplication.
    # Create two orchestrators with different global dedup modes.
    orchestrator_none = DepChainTaggerOrchestrator(
        patterns=(base_pattern, base_pattern),  # Duplicate pattern
        global_dedup_mode="none",
        max_total_matches=1000000,
    )
    orchestrator_dedup = DepChainTaggerOrchestrator(
        patterns=(base_pattern, base_pattern),  # Same duplicate pattern
        global_dedup_mode="role_based",
        max_total_matches=1000000,
    )

    matches_none = orchestrator_none.tag_sentence_layers(
        sentence_syntax_layers=sentence_layers,
        sentence_spans=sentence_spans,
    )
    matches_dedup = orchestrator_dedup.tag_sentence_layers(
        sentence_syntax_layers=sentence_layers,
        sentence_spans=sentence_spans,
    )

    # With global_dedup_mode='role_based', duplicate matches should be reduced.
    check(
        len(matches_dedup) <= len(matches_none),
        "Global role_based dedup should produce same or fewer matches than 'none' mode.",
    )

    # 7) max_total_matches should cap total matches across all sentences.
    orchestrator_capped = DepChainTaggerOrchestrator(
        patterns=(base_pattern,),
        global_dedup_mode="none",
        max_total_matches=1,  # Very restrictive cap
    )
    capped_matches = orchestrator_capped.tag_sentence_layers(
        sentence_syntax_layers=sentence_layers,
        sentence_spans=sentence_spans,
    )
    check(
        len(capped_matches) <= 1,
        "max_total_matches should cap results to specified limit.",
    )

    # 8) Constructor validation checks.
    assert_raises(
        TypeError,
        DepChainTaggerOrchestrator,
        patterns=[base_pattern],  # Should be tuple, not list
    )
    assert_raises(
        ValueError,
        DepChainTaggerOrchestrator,
        patterns=(base_pattern,),
        sentence_match_dedup_mode="invalid",
    )
    assert_raises(
        ValueError,
        DepChainTaggerOrchestrator,
        patterns=(base_pattern,),
        global_dedup_mode="invalid",
    )
    assert_raises(
        ValueError,
        DepChainTaggerOrchestrator,
        patterns=(base_pattern,),
        max_matches_per_sentence=0,
    )
    assert_raises(
        ValueError,
        DepChainTaggerOrchestrator,
        patterns=(base_pattern,),
        max_total_matches=0,
    )
    assert_raises(
        TypeError,
        DepChainTaggerOrchestrator,
        patterns=(base_pattern,),
        allow_role_node_overlap="no",
    )
    passed += 6
    total += 6

    # 9) tag_sentence_layers() should validate span/layer count alignment.
    assert_raises(
        ValueError,
        orchestrator.tag_sentence_layers,
        sentence_syntax_layers=sentence_layers,
        sentence_spans=[(0, 42)],  # Mismatched count
    )
    passed += 1
    total += 1

    print(f"DepChainTaggerOrchestrator tests passed: {passed}/{total}")


run_depchaintaggerorchestrator_tests()

DepChainTaggerOrchestrator tests passed: 18/18


In [28]:
# Testing DepChainTaggerOrchestrator on the sample text with the StanzaSyntaxTagger

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the DepChainTaggerOrchestrator implementation with the sample text
pattern = PathPattern(
    name="noun_to_adj",
    node_steps=(
        NodeConstraint(
            role="parent",
            upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
        ),
        NodeConstraint(
            role="child",
            upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
            deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "obj"),
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.UP,
            deprel_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
            min_hops=1,
            max_hops=1,
            allow_crossing_sentence=False,
        ),
    ),
    anchor_role="child",
    emit_roles=("parent", "child"),
)

orchestrator = DepChainTaggerOrchestrator(
    patterns=(pattern,),
    global_dedup_mode="none",
    max_total_matches=1000000,
)
decorated_matches = orchestrator.tag_and_decorate_sentence_layers(
    sentence_syntax_layers=[sent.stanza_syntax for sent in text_obj.sentences],
    sentence_spans=[(sent.start, sent.end) for sent in text_obj.sentences],
)
print(f"Found {len(decorated_matches)} decorated matches for pattern '{pattern.name}'.")
for row in decorated_matches:
    print("Decorated row:", row)
print("-" * 50)

Found 1 decorated matches for pattern 'noun_to_adj'.
Decorated row: {'pattern_name': 'noun_to_adj', 'sentence_index': 0, 'sentence_span': (0, 42), 'matched_text': 'lendurist abikaasale', 'role_to_token_id': {'parent': 3, 'child': 4}, 'role_to_text': {'parent': 'lendurist', 'child': 'abikaasale'}, 'role_to_span': {'parent': (9, 18), 'child': (19, 29)}, 'traversed_edges': [{'from_role': 'parent', 'to_role': 'child', 'direction': 'up', 'deprel': 'nmod', 'hops': 1, 'crosses_sentence': False}], 'metadata': {}}
--------------------------------------------------


In [88]:
sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

sentence_layer = text_obj.sentences
stanza_syntax_layer = text_obj.stanza_syntax

# Check the DepChainTaggerOrchestrator implementation with the sample text
pattern = PathPattern(
    name="noun_to_adj",
    node_steps=(
        NodeConstraint(
            role="parent",
            upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
        ),
        NodeConstraint(
            role="child",
            upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
            deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "obj"),
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.UP,
            deprel_condition=ValueCondition(ConditionMode.EXACT, "nmod"),
            min_hops=1,
            max_hops=1,
            allow_crossing_sentence=False,
        ),
    ),
    anchor_role="child",
    emit_roles=("parent", "child"),
)

orchestrator = DepChainTaggerOrchestrator(
    patterns=(pattern,),
    global_dedup_mode="none",
    max_total_matches=1000000,
)

for i, sentence in enumerate(sentence_layer):
    sentence_span = (sentence.start, sentence.end)
    print(f"Sentence span: {sentence_span}, text: '{sentence.text}'")
    stanza_syntax_layer_sentence = sentence.stanza_syntax
    # Build syntax graph index from stanza_syntax layer
    graph_index = SyntaxGraphIndex(
        stanza_syntax_layer=stanza_syntax_layer_sentence,
        sentence_id=i,
        sentence_span=sentence_span,
    )

    # Run matcher
    matcher = orchestrator._get_matcher()
    matches = matcher.match_sentence(
        graph_index=graph_index,
        sentence_index=i,
        sentence_span=sentence_span,
    )

    print(f"Found {len(matches)} matches in sentence {i}.")
    for match in matches:
        print(match.describe())
    print("-" * 50)

Sentence span: (0, 42), text: '['Ta', 'andis', 'lendurist', 'abikaasale', 'oma', 'raamatu', '.']'
Found 1 matches in sentence 0.
Pattern name: noun_to_adj
Sentence index: 0
Sentence span: (0, 42)
Matched roles:
  Role: child, Token ID: 4, Node properties: {start: 19, end: 29, upostag: S, xpostag: S, lemma: abikaasa, feats: OrderedDict([('sg', 'sg'), ('all', 'all')])}
  Role: parent, Token ID: 3, Node properties: {start: 9, end: 18, upostag: S, xpostag: S, lemma: lendur, feats: OrderedDict([('sg', 'sg'), ('el', 'el')])}
Traversed edges:
  From role: parent to role: child, Edge context: {direction: up, deprel: nmod, hops: 1, crosses_sentence: False}
Matched text: 'lendurist abikaasale'
--------------------------------------------------
Sentence span: (43, 70), text: '['See', 'raamat', 'on', 'väga', 'huvitav', '.']'
Found 0 matches in sentence 1.
--------------------------------------------------


In [143]:
from scripts.DepChainTagger import (
    ConditionMode,
    DepChainMatcher,
    DepChainTaggerOrchestrator,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    PhraseDecorator,
    SyntaxGraphIndex,
    ValueCondition,
    DepChainTagger,
)

# Testing DepChainTagger class and its EstNLTK-compatible wrapper on the sample text with the StanzaSyntaxTagger
pattern = PathPattern(
    name="noun_to_adj",
    node_steps=(
        NodeConstraint(
            role="parent",
            upostag_condition=ValueCondition(ConditionMode.MEMBERSHIP, ["S", "A"]),
        ),
        NodeConstraint(
            role="child",
            upostag_condition=ValueCondition(ConditionMode.EXACT, "S"),
            # deprel_at_node_condition=ValueCondition(ConditionMode.EXACT, "obj"),
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.BOTH,
            deprel_condition=ValueCondition(ConditionMode.WILDCARD),
            min_hops=1,
            max_hops=1,
            allow_crossing_sentence=False,
        ),
    ),
    anchor_role="parent",
    emit_roles=("parent", "child"),
)

tagger = DepChainTagger(
    patterns=(pattern,),
    sentence_match_dedup_mode="none",
    max_matches_per_sentence=1000000,
)

sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)
tagger.tag(text_obj)

Text(text='Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav.')

In [140]:
display(text_obj.dep_chains)

Layer(name='dep_chains', attributes=('pattern_name', 'matched_text', 'upostag', 'xpostag', 'feats', 'lemma', 'deprel', 'role', 'is_anchor', 'match_id'), spans=SL[Span('lendurist', [{'pattern_name': 'noun_to_adj', 'matched_text': 'lendurist abikaasale', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('el', 'el')]), 'lemma': 'lendur', 'deprel': 'nmod', 'role': 'parent', 'is_anchor': True, 'match_id': '0:noun_to_adj:-210991427258383335'}]),
Span('abikaasale', [{'pattern_name': 'noun_to_adj', 'matched_text': 'lendurist abikaasale', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('all', 'all')]), 'lemma': 'abikaasa', 'deprel': 'obj', 'role': 'child', 'is_anchor': False, 'match_id': '0:noun_to_adj:-210991427258383335'}]),
Span('raamat', [{'pattern_name': 'noun_to_adj', 'matched_text': 'huvitav raamat', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('n', 'n')]), 'lemma': 'raamat', 'deprel': 'nsubj:cop', 'role': 'child', 'is_anchor': False, 'match_id': '1:noun_to_adj:-7585524074862212047'}]),
Span('huvitav', [{'pattern_name': 'noun_to_adj', 'matched_text': 'huvitav raamat', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('sg', 'sg'), ('n', 'n')]), 'lemma': 'huvitav', 'deprel': 'root', 'role': 'parent', 'is_anchor': True, 'match_id': '1:noun_to_adj:-7585524074862212047'}])])

In [144]:
text_obj.dep_chains.display()

Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav .